### ResNet-50
Réseau CNN profond à 50 couches avec des skip connections pour éviter la disparition du gradient lors de l'entraînement.

### EfficientNet-B0
Modèle compact qui optimise précision et efficacité. Plus léger que ResNet pour des performances similaires.

### VGG-16
Architecture séquentielle simple à 16 couches. Facile à comprendre mais lourde (~138M paramètres).

### ViT — Vision Transformer
Découpe l'image en patches et utilise un mécanisme d'attention. Très performant sur de grandes bases de données.

In [1]:
import torch
import torchvision
import transformers

print("PyTorch version :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
print("Transformers version :", transformers.__version__)

PyTorch version : 2.10.0+cpu
GPU disponible : False
Transformers version : 5.3.0


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé  : {DEVICE}")

PyTorch version : 2.10.0+cpu
CUDA disponible : False
Device utilisé  : cpu


Chargement des données

In [4]:
import kagglehub

dataset_path = kagglehub.dataset_download("mohamedhanyyy/chest-ctscan-images")
DATA_ROOT = Path(dataset_path) / "Data"

CLASS_NAMES  = ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']
CLASS_LABELS = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print(f"✅ Dataset chargé : {DATA_ROOT}")
print(f"Classes : {CLASS_NAMES}")

✅ Dataset chargé : C:\Users\camil\.cache\kagglehub\datasets\mohamedhanyyy\chest-ctscan-images\versions\1\Data
Classes : ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']


Préparer les transformations (224x224) 

In [5]:
transform = T.Compose([
    T.Resize((224, 224)),       # Redimensionner
    T.Grayscale(num_output_channels=3),  # CT-scan = niveaux de gris → forcer 3 canaux
    T.ToTensor(),               # Convertir en tenseur [0,1]
    T.Normalize(                # Normaliser comme ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Transformations définies")
print("   Resize → 224x224")
print("   Grayscale → 3 canaux")
print("   Normalize → valeurs ImageNet")

✅ Transformations définies
   Resize → 224x224
   Grayscale → 3 canaux
   Normalize → valeurs ImageNet


Création du dataset 

In [6]:
from torch.utils.data import Dataset, DataLoader

class CTScanDataset(Dataset):
    def __init__(self, data_root, split, transform=None):
        self.transform = transform
        self.samples = []
        
        split_dir = Path(data_root) / split
        for folder in sorted(split_dir.iterdir()):
            if not folder.is_dir():
                continue
            # Trouver la classe correspondante
            class_name = None
            for cls in CLASS_NAMES:
                if folder.name == cls or folder.name.startswith(cls + '_'):
                    class_name = cls
                    break
            if class_name is None:
                continue
            label = CLASS_LABELS[class_name]
            for img_path in folder.glob('*.*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    self.samples.append((str(img_path), label))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Créer train / valid / test
train_dataset = CTScanDataset(DATA_ROOT, 'train', transform)
valid_dataset = CTScanDataset(DATA_ROOT, 'valid', transform)
test_dataset  = CTScanDataset(DATA_ROOT, 'test',  transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"✅ Dataset prêt !")
print(f"   Train : {len(train_dataset)} images")
print(f"   Valid : {len(valid_dataset)} images")
print(f"   Test  : {len(test_dataset)} images")

✅ Dataset prêt !
   Train : 613 images
   Valid : 72 images
   Test  : 315 images


In [7]:
from torchvision import models

# Charger ResNet-50 pré-entraîné sur ImageNet
model = models.resnet50(weights="IMAGENET1K_V1")

# Remplacer la dernière couche : 1000 classes → 4 classes
nb_classes = len(CLASS_NAMES)  # 4
model.fc = nn.Linear(model.fc.in_features, nb_classes)

# Envoyer sur le bon device
model = model.to(DEVICE)

# Compter les paramètres
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ ResNet-50 chargé avec succès !")
print(f"   Paramètres totaux      : {total_params:,}")
print(f"   Paramètres entraînables: {trainable_params:,}")
print(f"   Dernière couche        : {model.fc}")
print(f"   Device                 : {DEVICE}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\camil/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|█████████████████████| 97.8M/97.8M [00:45<00:00, 2.27MB/s]


✅ ResNet-50 chargé avec succès !
   Paramètres totaux      : 23,516,228
   Paramètres entraînables: 23,516,228
   Dernière couche        : Linear(in_features=2048, out_features=4, bias=True)
   Device                 : cpu


Test avec une image 

In [8]:
# image de test 
sample_path, sample_label = test_dataset.samples[0]
img_tensor = test_dataset[0][0].unsqueeze(0).to(DEVICE)  

# Passer dans le modèle
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probabilities = torch.softmax(output, dim=1)[0]
    predicted_idx = probabilities.argmax().item()

print("✅ Test d'inférence réussi !")
print(f"   Image testée   : {Path(sample_path).name}")
print(f"   Vraie classe   : {CLASS_NAMES[sample_label]}")
print(f"   Classe prédite : {CLASS_NAMES[predicted_idx]}")
print(f"\n   Scores par classe :")
for cls, prob in zip(CLASS_NAMES, probabilities):
    bar = "█" * int(prob * 30)
    print(f"   {cls:<30} {bar} {prob:.1%}")

✅ Test d'inférence réussi !
   Image testée   : 000108 (3).png
   Vraie classe   : adenocarcinoma
   Classe prédite : large.cell.carcinoma

   Scores par classe :
   adenocarcinoma                 ███████ 24.9%
   large.cell.carcinoma           █████████ 31.0%
   normal                         ██████ 22.0%
   squamous.cell.carcinoma        ██████ 22.2%


Le modèle ResNet-50 pré-entraîné a été chargé  et testé sur une image CT-scan d'adénocarcinome, produisant un score de confiance de 31% pour la classe large.cell.carcinoma. C'est une prédiction incorrecte attendue car le modèle n'a pas encore été entrainé sur les données médicales.